# Phase 2: DeBERTa encoder candidate

This is an isolated experiment. It does not modify the production Qwen+SVC baseline. The model extracts frozen DeBERTa-v3-base representations and fits a weighted LinearSVC head. Official competition rows receive weight 1.0; the public pool receives a conservative 0.15 weight to reduce protocol-shift damage.

In [ ]:
import gc, glob, json, os, time
from pathlib import Path
import numpy as np, pandas as pd, torch
from transformers import AutoModel, AutoTokenizer, AutoConfig
from sklearn.svm import LinearSVC
T0=time.time(); INPUT_DIR="/kaggle/input"
LABELS=['gpt-5.5','gpt-5.6-luna','gpt-5.6-sol','gpt-5.6-terra','gpt-6-astra','gemini-3.1-pro-preview','gemini-3.6-flash','gemini-3.7-flash','gemini-3.8-flash','claude-opus-4-6','claude-opus-4-8','claude-opus-5','claude-sonnet-5','claude-fable-5','qwen3.8-max','glm-5.3','glm-5.3-flash','kimi-k3']
L2I={x:i for i,x in enumerate(LABELS)}; K=len(LABELS)
def find(pattern):
 hits=sorted(glob.glob(f"{INPUT_DIR}/**/{pattern}",recursive=True))
 if not hits: raise FileNotFoundError(pattern)
 return hits[0]
def safe_prob(z):
 z=np.asarray(z,dtype=np.float64)
 if z.ndim!=2 or z.shape[1]!=K or not np.isfinite(z).all(): raise ValueError("invalid score matrix")
 z-=z.max(1,keepdims=True); e=np.exp(np.clip(z,-50,50)); p=e/e.sum(1,keepdims=True)
 if not np.isfinite(p).all(): raise FloatingPointError("invalid probabilities")
 return p
train=pd.read_csv(find('train.csv')); test=pd.read_csv(find('test.csv'))
if not {'id','prompt_id','text','model'}.issubset(train.columns) or not {'id','text'}.issubset(test.columns): raise ValueError('schema mismatch')
if train.id.duplicated().any() or test.id.duplicated().any(): raise ValueError('duplicate ids')
if set(train.model)-set(LABELS): raise ValueError('unknown labels')
train.text=train.text.fillna('').astype(str); test.text=test.text.fillna('').astype(str)
y=train.model.map(L2I).to_numpy(dtype=np.int64)
pool=pd.read_parquet(find('pool_v1.parquet')).dropna(subset=['text','model']).copy()
if set(pool.model)-set(LABELS): raise ValueError('unknown pool labels')
pool.text=pool.text.astype(str); py=pool.model.map(L2I).to_numpy(dtype=np.int64)
print('train',train.shape,'pool',pool.shape,'test',test.shape,flush=True)


In [ ]:
# Locate the offline Kaggle-packaged DeBERTa-v3-base model by its config, not by a guessed mount path.
base=None
for cfg_path in sorted(glob.glob(f"{INPUT_DIR}/**/config.json",recursive=True)):
 try: cfg=json.load(open(cfg_path))
 except Exception: continue
 if cfg.get('model_type') in {'deberta-v2','deberta'} and cfg.get('hidden_size')==768 and cfg.get('num_hidden_layers')==12:
  base=os.path.dirname(cfg_path); break
if base is None: raise FileNotFoundError('DeBERTa-v3-base model input not found')
print('deberta base',base,flush=True)
tok=AutoTokenizer.from_pretrained(base); model=AutoModel.from_pretrained(base,torch_dtype=torch.float16).to('cuda:0').eval()
@torch.no_grad()
def embed(texts,bs=32):
 out=np.zeros((len(texts),model.config.hidden_size),np.float32)
 order=np.argsort([len(x) for x in texts],kind='stable')
 for st in range(0,len(order),bs):
  idx=order[st:st+bs]; enc=tok([texts[i] for i in idx],padding=True,truncation=True,max_length=512,return_tensors='pt').to('cuda:0')
  hs=model(**enc).last_hidden_state.float(); mask=enc['attention_mask'].unsqueeze(-1).float(); z=(hs*mask).sum(1)/mask.sum(1).clamp(min=1); out[idx]=z.cpu().numpy()
 return out
all_train=train.text.tolist()+pool.text.tolist(); X=embed(all_train); Xt=embed(test.text.tolist()); del model,tok; torch.cuda.empty_cache(); gc.collect()
if not np.isfinite(X).all() or not np.isfinite(Xt).all(): raise FloatingPointError('non-finite embeddings')
print('embeddings',X.shape,Xt.shape,'elapsed',time.time()-T0,flush=True)


In [ ]:
# Conservative pool weighting; official train dominates the classifier head.
fit_y=np.r_[y,py]; sample_weight=np.r_[np.ones(len(train),np.float32),np.full(len(pool),0.15,np.float32)]
clf=LinearSVC(C=0.5,class_weight='balanced',max_iter=3000,random_state=42)
clf.fit(X,fit_y,sample_weight=sample_weight)
P=safe_prob(clf.decision_function(Xt))
pred=np.array(LABELS)[P.argmax(1)]
out=pd.DataFrame({'id':test.id.astype(str),'model':pred})
if len(out)!=len(test) or out.id.duplicated().any() or set(out.id)!=set(test.id.astype(str)): raise ValueError('submission contract failure')
if set(out.model)-set(LABELS): raise ValueError('invalid prediction label')
out.to_parquet('deberta_phase2.parquet',index=False)
Path('deberta_phase2_summary.json').write_text(json.dumps({'train_rows':len(train),'pool_rows':len(pool),'pool_weight':0.15,'hidden_size':int(X.shape[1]),'test_rows':len(test),'runtime_seconds':time.time()-T0},indent=2))
print('phase2 output',out.shape,'runtime',time.time()-T0,flush=True)
